In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('resume_extracted.csv')
selected_categories = [
    "INFORMATION-TECHNOLOGY",
    "BUSINESS-DEVELOPMENT",
    "ACCOUNTANT",
    "ENGINEERING",
    "FINANCE",
    "BANKING",
    "HEALTHCARE",
    "DESIGNER",
    "TEACHER",
    "HR"
]
df = df[df["category"].isin(selected_categories)]
df.dropna(subset=['resume_text','category'],inplace=True)
df = df.reset_index(drop=True)
df=df.drop_duplicates(subset=['resume_text'])
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words=set(stopwords.words('english'))
keep_words = {'no', 'nor', 'not', 'without', 'against', 'only', 'but'}
stop_words=stop_words-keep_words
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')
import re
import html


def data_preprocessing(text):
    text = html.unescape(text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s+#./-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'\b\d{10,}\b', ' ', text)
    text = re.sub(r'\.(?=\s|$)', ' ', text)
    text=text.lower()
    if len(text)<50:
        return ""
    tokens=text.split()
    new_tokens=[]
    for word in tokens:
        if word not in stop_words:
            new_tokens.append(word)
    tokens=new_tokens

    lemmatizer = WordNetLemmatizer()

    def get_wordnet_pos(tag):
        if tag.startswith('J'):
            return wordnet.ADJ
        elif tag.startswith('V'):
            return wordnet.VERB
        elif tag.startswith('N'):
            return wordnet.NOUN
        elif tag.startswith('R'):
            return wordnet.ADV
        return wordnet.NOUN

    pos_tags = nltk.pos_tag(tokens)

    lemmatized_tokens = []

    for word, tag in pos_tags:
        pos = get_wordnet_pos(tag)
        lemmatized_word = lemmatizer.lemmatize(word, pos)
        lemmatized_tokens.append(lemmatized_word)

    tokens = lemmatized_tokens 
    return ' '.join(tokens)
# from sklearn.preprocessing import FunctionTransformer
# preprocessor=FunctionTransformer(
#     lambda X:X.apply(data_preprocessing),
#     validate=False
# )

df['resume_text']=df['resume_text'].apply(data_preprocessing)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hites\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\hites\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hites\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hites\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
df.head()

,filename,resume_text,category
0,10554236.pdf,accountant summary financial accountant specia...,ACCOUNTANT
1,10674770.pdf,staff accountant summary highly analytical det...,ACCOUNTANT
2,11163645.pdf,accountant professional summary obtain positio...,ACCOUNTANT
3,11759079.pdf,senior accountant experience company name june...,ACCOUNTANT
4,12065211.pdf,senior accountant professional summary senior ...,ACCOUNTANT


In [4]:
df['category'].value_counts()

category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      119
ACCOUNTANT                118
ENGINEERING               118
FINANCE                   117
BANKING                   115
HEALTHCARE                115
HR                        110
DESIGNER                  107
TEACHER                   102
Name: count, dtype: int64

In [5]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import loguniform

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
model=Pipeline([('tfidf',TfidfVectorizer(max_features=10000,ngram_range=(1,2),sublinear_tf=True)),('classifier',LinearSVC(C=7.915074397656218,class_weight='balanced',random_state=42))])
X=df['resume_text']
y=df['category']
from sklearn.model_selection import train_test_split,GridSearchCV
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


In [7]:
# param_dist = {
#     "classifier__C": loguniform(0.001, 100),
#     "tfidf__max_features": [10000, 15000, 20000, 30000],
#     "tfidf__ngram_range": [(1, 1), (1, 2)],
#     "tfidf__min_df": [1, 2, 3, 5]
# }

# # Randomized Search
# random_search = RandomizedSearchCV(
#     estimator=model,
#     param_distributions=param_dist,
#     n_iter=15,
#     cv=3,
#     scoring="f1_macro",
#     random_state=42,
#     n_jobs=1,
#     verbose=2,
#     error_score="raise"
# )

In [8]:
#Best Parameters are :  {'classifier__C': np.float64(7.915074397656218), 'tfidf__max_features': 10000, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 2)}

In [9]:
model.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](10,)","['ACCOUNTANT','BANKING','BUSINESS-DEVELOPMENT',...,'HR', 'INFORMATION-TECHNOLOGY','TEACHER']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",10000
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'


In [10]:
# print("Best Parameters are : ",random_search.best_params_)

In [11]:
y_pred=model.predict(X_test)

In [12]:
from sklearn.metrics import accuracy_score,precision_score,recall_score
print(accuracy_score(y_test,y_pred))
print("Precision:", precision_score(
    y_test, y_pred, average='weighted', zero_division=0
))

print("Recall:", recall_score(
    y_test, y_pred, average='weighted', zero_division=0
))

0.8820960698689956
Precision: 0.8860117101288547
Recall: 0.8820960698689956


In [13]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, zero_division=0))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.78      0.88      0.82        24
               BANKING       0.90      0.83      0.86        23
  BUSINESS-DEVELOPMENT       0.91      0.88      0.89        24
              DESIGNER       0.88      1.00      0.93        21
           ENGINEERING       0.96      0.92      0.94        24
               FINANCE       0.85      0.71      0.77        24
            HEALTHCARE       0.95      0.83      0.88        23
                    HR       0.81      1.00      0.90        22
INFORMATION-TECHNOLOGY       0.88      0.88      0.88        24
               TEACHER       0.95      0.95      0.95        20

              accuracy                           0.88       229
             macro avg       0.89      0.89      0.88       229
          weighted avg       0.89      0.88      0.88       229



In [15]:
import joblib

joblib.dump(model, "resume_screening_model.pkl")

['resume_screening_model.pkl']

In [16]:
resume_text = """
Dedicated mathematics teacher with a B.Ed. and M.Sc. in Mathematics.
Experienced in classroom management, lesson planning, curriculum
development, student assessment, and academic support.
Skilled in educational technology, smart boards, and online teaching.
"""

In [18]:
model.predict([resume_text])

array(['TEACHER'], dtype=object)

In [21]:
resume_3 = """
Human Resources Manager with experience in recruitment,
employee relations, talent acquisition, and performance management.
Skilled in conducting interviews, screening resumes, onboarding
employees, payroll administration, and workforce planning.
Experienced in HR policies, employee engagement, conflict resolution,
training coordination, and maintaining employee records.
"""

In [22]:
model.predict([resume_3])

array(['HR'], dtype=object)

In [23]:
resume_5 = """
Experienced English teacher with a Bachelor of Education degree.
Skilled in classroom management, lesson planning, curriculum design,
student assessment, and educational psychology.
Experienced in teaching grammar, literature, reading comprehension,
and writing skills. Capable of preparing examinations, evaluating
student performance, and creating engaging classroom activities.
"""

In [24]:
model.predict([resume_5])

array(['TEACHER'], dtype=object)

In [25]:
resume_2 = """
Information Technology professional and Software Developer with
experience in software engineering and backend application development.

Technical Skills:
Python, Java, C++, SQL, MySQL, FastAPI, Django, REST APIs,
data structures, algorithms, object-oriented programming,
software testing, debugging, Git and GitHub.

Work Experience:
Developed and maintained software applications, implemented
backend services, designed database schemas, created REST APIs,
fixed software bugs, and optimized application performance.

Projects:
Built Python applications, developed backend APIs using FastAPI,
integrated SQL databases, and implemented machine learning models.
Experienced in programming, software development, and computer systems.
"""

print(model.predict([resume_2])[0])

INFORMATION-TECHNOLOGY


In [26]:
resume_accountant = """
Experienced Accountant with a strong background in financial
management, bookkeeping, accounting operations, and financial reporting.

Technical Skills:
Accounting, financial analysis, bookkeeping, taxation, auditing,
Microsoft Excel, Tally ERP, QuickBooks, accounts payable,
accounts receivable, bank reconciliation, payroll processing,
budget preparation, balance sheets, income statements,
general ledger, and financial statements.

Work Experience:
Worked as an Accountant responsible for maintaining financial records,
preparing monthly financial statements, reconciling bank accounts,
processing invoices, managing company expenses, and preparing tax
returns. Performed financial audits and ensured compliance with
accounting standards and tax regulations.

Education:
Bachelor of Commerce (B.Com.) in Accounting and Finance.

Responsibilities:
Managed daily accounting transactions, prepared balance sheets,
maintained general ledger entries, calculated taxes, analyzed financial
data, processed payroll, and assisted management with budgeting
and financial planning.

Soft Skills:
Attention to detail, analytical thinking, problem-solving,
time management, organization, and communication.
"""

prediction = model.predict([resume_accountant])

print("Predicted category:", prediction[0])

Predicted category: ACCOUNTANT
